# BoxBunny — Master Runner Notebook

This notebook contains all commands needed to build, run, test, and manage the BoxBunny boxing training robot system.

**Demo Users:**
| User | Password | Level | Sessions | Rank |
|------|----------|-------|----------|------|
| alex | boxing123 | Beginner | 8 | Novice |
| maria | boxing123 | Intermediate | 35 | Fighter |
| jake | boxing123 | Advanced | 120 | Champion |
| sarah | coaching123 | Coach | 3 coaching | — |

---
## 0. Environment Setup
Run this cell first to set up the environment.

In [ ]:
%%bash
# Source ROS 2 and workspace
echo "=== Environment Setup ==="
source /opt/ros/humble/setup.bash
if [ -f install/setup.bash ]; then
    source install/setup.bash
    echo "Workspace overlay sourced"
fi
echo "ROS_DISTRO: $ROS_DISTRO"
echo "Python: $(python3 --version)"
echo "Working dir: $(pwd)"
echo ""
echo "=== Package Status ==="
ros2 pkg list 2>/dev/null | grep boxbunny || echo "(packages not built yet — run Build cell first)"

---
## 1. Build Workspace
Builds all 4 ROS 2 packages: boxbunny_msgs, boxbunny_core, boxbunny_gui, boxbunny_dashboard

In [ ]:
%%bash
source /opt/ros/humble/setup.bash
echo "=== Building ROS 2 Workspace ==="
colcon build --symlink-install 2>&1
echo ""
echo "=== Build Complete ==="
source install/setup.bash
echo "Packages built:"
ros2 pkg list 2>/dev/null | grep boxbunny

---
## 2. Verify Messages
Check that all custom ROS 2 messages and services are built correctly.

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Custom Messages ==="
ros2 interface list | grep boxbunny_msgs | head -30
echo ""
echo "=== Sample Message: ConfirmedPunch ==="
ros2 interface show boxbunny_msgs/msg/ConfirmedPunch
echo ""
echo "=== Sample Service: StartSession ==="
ros2 interface show boxbunny_msgs/srv/StartSession

---
## 3. Run Tests
Runs the full pytest suite (171 tests — no hardware needed).

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
python3 -m pytest tests/ -v --tb=short 2>&1

---
## 4. Seed Demo Data
Creates demo users with realistic training history. Safe to re-run (idempotent).

In [ ]:
%%bash
python3 tools/demo_data_seeder.py

---
## 5. Launch System — Development Mode
Launches all ROS nodes + IMU simulator + GUI. Use this for development/testing.

**Note:** This launches in the background. Use the Stop cell below to shut down.

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in dev mode..."
echo "(IMU simulator will open in a separate window)"
echo ""
ros2 launch boxbunny_core boxbunny_dev.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo "Run the 'Stop System' cell to shut down"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
echo ""
echo "=== Active ROS Nodes ==="
ros2 node list 2>/dev/null || echo "(nodes still starting...)"

---
## 6. Launch System — Full Production Mode
Launches everything including real hardware connections.

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching BoxBunny in FULL mode..."
ros2 launch boxbunny_core boxbunny_full.launch.py &
LAUNCH_PID=$!
echo "Launch PID: $LAUNCH_PID"
echo $LAUNCH_PID > /tmp/boxbunny_launch.pid
sleep 5
ros2 node list 2>/dev/null

---
## 7. Stop System
Shuts down all BoxBunny ROS nodes.

In [ ]:
%%bash
if [ -f /tmp/boxbunny_launch.pid ]; then
    PID=$(cat /tmp/boxbunny_launch.pid)
    kill $PID 2>/dev/null && echo "Stopped launch process $PID" || echo "Process already stopped"
fi
# Kill any remaining boxbunny nodes
pkill -f 'boxbunny_core' 2>/dev/null
pkill -f 'boxbunny_gui' 2>/dev/null
pkill -f 'boxbunny_dashboard' 2>/dev/null
pkill -f 'imu_simulator' 2>/dev/null
echo "All BoxBunny processes stopped"

---
## 8. Launch Individual Components
Run specific subsystems for isolated testing.

### 8a. IMU Simulator Only

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Starting IMU Simulator..."
python3 tools/imu_simulator.py &
sleep 2
echo "IMU Simulator running. Click pads to publish ROS topics."

### 8b. GUI Only

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
export QT_QPA_PLATFORM=xcb
echo "Starting BoxBunny GUI..."
python3 -m boxbunny_gui.app &
sleep 2
echo "GUI launched"

### 8c. Dashboard Server Only

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Starting Dashboard Server on http://localhost:8080 ..."
python3 -m boxbunny_dashboard.server &
sleep 3
echo "Dashboard running at http://localhost:8080"
echo "Login with: alex / boxing123 (or maria, jake, sarah)"

### 8d. Headless Nodes Only (no GUI)

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Launching headless nodes..."
ros2 launch boxbunny_core headless.launch.py &
sleep 3
ros2 node list 2>/dev/null

---
## 9. Monitor ROS Topics
View live data flowing through the system.

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "=== Active ROS Topics ==="
ros2 topic list 2>/dev/null | grep boxbunny | sort
echo ""
echo "=== Active Nodes ==="
ros2 node list 2>/dev/null | grep -v _ros2cli | sort
echo ""
echo "=== Topic Hz (punch confirmed) ==="
timeout 3 ros2 topic hz /boxbunny/punch/confirmed 2>/dev/null || echo "(no data yet)"

### 9a. Echo a specific topic (5 messages)

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
# Change topic name as needed:
TOPIC="/boxbunny/imu/nav_event"
echo "Listening to $TOPIC (5 messages, 10s timeout)..."
timeout 10 ros2 topic echo $TOPIC --once 2>/dev/null || echo "(no messages received — is a publisher running?)"

---
## 10. Database Inspection
View demo user data in the databases.

In [ ]:
import sqlite3, json

# Main database
conn = sqlite3.connect('data/boxbunny_main.db')
conn.row_factory = sqlite3.Row

print("=== Users ===")
for row in conn.execute("SELECT id, username, display_name, user_type, level, last_login FROM users"):
    print(f"  {dict(row)}")

print("\n=== Presets ===")
for row in conn.execute("SELECT user_id, name, preset_type, is_favorite, use_count FROM presets LIMIT 10"):
    print(f"  {dict(row)}")

print("\n=== Coaching Sessions ===")
for row in conn.execute("SELECT * FROM coaching_sessions"):
    print(f"  {dict(row)}")

print("\n=== Guest Sessions ===")
for row in conn.execute("SELECT * FROM guest_sessions"):
    print(f"  {dict(row)}")

conn.close()

In [ ]:
import sqlite3, json

# Per-user database — change username as needed
USERNAME = "maria"  # Try: alex, maria, jake

db_path = f'data/users/{USERNAME}/boxbunny.db'
conn = sqlite3.connect(db_path)
conn.row_factory = sqlite3.Row

print(f"=== {USERNAME}'s Training Sessions (last 10) ===")
for row in conn.execute("SELECT session_id, mode, difficulty, rounds_completed, is_complete FROM training_sessions ORDER BY started_at DESC LIMIT 10"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s XP & Rank ===")
for row in conn.execute("SELECT * FROM user_xp"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Streak ===")
for row in conn.execute("SELECT * FROM streaks"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Achievements ===")
for row in conn.execute("SELECT * FROM achievements"):
    print(f"  {dict(row)}")

print(f"\n=== {USERNAME}'s Personal Records ===")
for row in conn.execute("SELECT * FROM personal_records"):
    print(f"  {dict(row)}")

conn.close()

---
## 11. Model Verification
Check that all ML models are present and loadable.

In [ ]:
import os, sys
sys.path.insert(0, '.')

models = {
    'CV Action Model (.pth)': 'action_prediction/model/best_model.pth',
    'CV Action Model (.onnx)': 'action_prediction/model/best_model.onnx',
    'CV Action Model (.trt)': 'action_prediction/model/best_model.trt',
    'YOLO Pose Model': 'action_prediction/model/yolo26n-pose.pt',
    'LLM (Qwen2.5-3B)': 'models/llm/qwen2.5-3b-instruct-q4_k_m.gguf',
}

print("=== Model Status ===")
for name, path in models.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  OK  {name}: {size_mb:.1f} MB")
    else:
        print(f"  MISSING  {name}: {path}")

# Test CV model loading
print("\n=== CV Model Load Test ===")
try:
    import torch
    ckpt = torch.load('action_prediction/model/best_model.pth', map_location='cpu', weights_only=False)
    config = ckpt.get('config', {})
    print(f"  Model loaded successfully")
    print(f"  Classes: {ckpt.get('label_names', 'default 8-class')}")
    print(f"  Config keys: {list(config.keys())[:10]}")
except Exception as e:
    print(f"  Load failed: {e}")

---
## 12. LLM Test
Test the local LLM model (Qwen2.5-3B).

In [ ]:
import time

MODEL_PATH = 'models/llm/qwen2.5-3b-instruct-q4_k_m.gguf'

try:
    from llama_cpp import Llama
    print("Loading LLM (this takes 10-30 seconds)...")
    start = time.time()
    llm = Llama(model_path=MODEL_PATH, n_gpu_layers=-1, n_ctx=512, verbose=False)
    load_time = time.time() - start
    print(f"Model loaded in {load_time:.1f}s")
    
    # Generate a coaching tip
    print("\nGenerating coaching tip...")
    start = time.time()
    result = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": "You are BoxBunny AI Coach, an expert boxing trainer. Keep responses under 2 sentences."},
            {"role": "user", "content": "I just did a training session: 87 punches, mostly jabs and crosses, accuracy 72%. Give me a quick tip."},
        ],
        max_tokens=80,
        temperature=0.7,
    )
    gen_time = time.time() - start
    tip = result['choices'][0]['message']['content']
    print(f"AI Coach says ({gen_time:.1f}s): {tip}")
    
    del llm  # Free GPU memory
except ImportError:
    print("llama-cpp-python not installed. Run: pip install llama-cpp-python")
except Exception as e:
    print(f"LLM test failed: {e}")

---
## 13. Camera Test
Test the RealSense D435i camera connection.

In [ ]:
try:
    import pyrealsense2 as rs
    ctx = rs.context()
    devices = ctx.query_devices()
    if len(devices) > 0:
        dev = devices[0]
        print(f"Camera found: {dev.get_info(rs.camera_info.name)}")
        print(f"Serial: {dev.get_info(rs.camera_info.serial_number)}")
        print(f"Firmware: {dev.get_info(rs.camera_info.firmware_version)}")
    else:
        print("No RealSense camera detected (connect D435i or use recorded data)")
except ImportError:
    print("pyrealsense2 not installed")
except Exception as e:
    print(f"Camera check failed: {e}")

---
## 14. Action Prediction — Standalone Test
Test the CV model inference without ROS (uses camera directly).

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
echo "Running action prediction standalone test..."
echo "(Press Ctrl+C or close the window to stop)"
python3 action_prediction/run.py \
    --checkpoint action_prediction/model/best_model.pth \
    --pose-weights action_prediction/model/yolo26n-pose.pt \
    2>&1 || echo "(camera not available — connect D435i to run this test)"

---
## 15. Dashboard Server Test
Start the phone dashboard and test API endpoints.

In [ ]:
%%bash
source /opt/ros/humble/setup.bash && source install/setup.bash
# Start dashboard in background
python3 -m boxbunny_dashboard.server &
sleep 3

echo "=== Testing API ==="
echo ""
echo "--- Health Check ---"
curl -s http://localhost:8080/api/health | python3 -m json.tool 2>/dev/null || echo "(server not responding)"

echo ""
echo "--- Login as Alex ---"
TOKEN=$(curl -s -X POST http://localhost:8080/api/auth/login \
    -H 'Content-Type: application/json' \
    -d '{"username": "alex", "password": "boxing123"}' | python3 -c "import sys,json; print(json.load(sys.stdin).get('token',''))" 2>/dev/null)
echo "Token: ${TOKEN:0:20}..."

if [ -n "$TOKEN" ]; then
    echo ""
    echo "--- Alex's Session History ---"
    curl -s http://localhost:8080/api/sessions/history \
        -H "Authorization: Bearer $TOKEN" | python3 -m json.tool 2>/dev/null | head -30

    echo ""
    echo "--- Alex's Gamification Profile ---"
    curl -s http://localhost:8080/api/gamification/profile \
        -H "Authorization: Bearer $TOKEN" | python3 -m json.tool 2>/dev/null
fi

# Stop dashboard
pkill -f 'boxbunny_dashboard.server' 2>/dev/null
echo ""
echo "Dashboard test complete"

---
## 16. Build Vue Frontend
Rebuild the phone dashboard Vue 3 SPA.

In [ ]:
%%bash
export NVM_DIR="$HOME/.nvm"
[ -s "$NVM_DIR/nvm.sh" ] && . "$NVM_DIR/nvm.sh"
cd src/boxbunny_dashboard/frontend
echo "Node: $(node --version), npm: $(npm --version)"
echo ""
echo "=== Installing dependencies ==="
npm install 2>&1 | tail -3
echo ""
echo "=== Building Vue SPA ==="
npm run build 2>&1
echo ""
echo "=== Output ==="
ls -lh ../static/dist/ 2>/dev/null | head -5

---
## 17. Download Models
Download LLM model and check all model files.

In [ ]:
%%bash
bash scripts/download_models.sh

---
## 18. Full Setup (First Time)
One-command bootstrap for a fresh system. Installs deps, builds, downloads, seeds.

In [ ]:
%%bash
bash scripts/setup.sh 2>&1

---
## 19. Hardware Check
Quick status check of all hardware components.

In [ ]:
import os, sys

checks = [
    ('RealSense Camera', lambda: __import__('pyrealsense2').context().query_devices().size() > 0),
    ('PyTorch', lambda: __import__('torch').__version__),
    ('CUDA Available', lambda: __import__('torch').cuda.is_available()),
    ('PySide6', lambda: __import__('PySide6').__version__),
    ('FastAPI', lambda: __import__('fastapi').__version__),
    ('MediaPipe', lambda: __import__('mediapipe').__version__),
    ('llama-cpp-python', lambda: __import__('llama_cpp').__version__ if hasattr(__import__('llama_cpp'), '__version__') else 'installed'),
    ('bcrypt', lambda: __import__('bcrypt').__version__),
    ('qrcode', lambda: 'installed' if __import__('qrcode') else ''),
    ('CV Model', lambda: os.path.exists('action_prediction/model/best_model.pth')),
    ('YOLO Model', lambda: os.path.exists('action_prediction/model/yolo26n-pose.pt')),
    ('LLM Model', lambda: os.path.exists('models/llm/qwen2.5-3b-instruct-q4_k_m.gguf')),
    ('Main DB', lambda: os.path.exists('data/boxbunny_main.db')),
]

print("=== Hardware & Dependency Check ===")
for name, check_fn in checks:
    try:
        result = check_fn()
        if isinstance(result, bool):
            status = 'OK' if result else 'MISSING'
        else:
            status = f'OK ({result})'
    except Exception as e:
        status = f'MISSING ({type(e).__name__})'
    symbol = '+' if 'OK' in status else '-'
    print(f"  [{symbol}] {name}: {status}")

---
## 20. Project Statistics

In [ ]:
%%bash
echo "=== BoxBunny Project Stats ==="
echo "Python files: $(find . -name '*.py' -not -path './_archive/*' -not -path './build/*' -not -path './install/*' -not -path '*/__pycache__/*' | wc -l)"
echo "Vue/JS files: $(find . \( -name '*.vue' -o -name '*.js' \) -not -path '*/node_modules/*' -not -path './_archive/*' -not -path './build/*' | wc -l)"
echo "ROS messages: $(find src/boxbunny_msgs -name '*.msg' | wc -l)"
echo "ROS services: $(find src/boxbunny_msgs -name '*.srv' | wc -l)"
echo "GUI widgets:  $(find src/boxbunny_gui/boxbunny_gui/widgets -name '*.py' -not -name '__init__.py' | wc -l)"
echo "GUI pages:    $(find src/boxbunny_gui/boxbunny_gui/pages -name '*.py' -not -name '__init__.py' | wc -l)"
echo "ROS nodes:    $(find src/boxbunny_core/boxbunny_core -name '*_node.py' -o -name '*_manager.py' -o -name '*_engine.py' -o -name '*_processor.py' | wc -l)"
echo "API endpoints: $(find src/boxbunny_dashboard/boxbunny_dashboard/api -name '*.py' -not -name '__init__.py' | wc -l)"
echo "Knowledge docs: $(find data/boxing_knowledge -type f | wc -l)"
echo "Test files:    $(find tests -name 'test_*.py' | wc -l)"
echo "Config files:  $(find config -type f | wc -l)"
echo "Launch files:  $(find src -name '*.launch.py' | wc -l)"

---
## 21. Sound Test
Play each sound effect to verify audio output. The first cell lists all available sounds; the second plays them all in sequence.